# Advanced RAG from first principles

This notebook builds an inspectable Advanced RAG pipeline over the same fictional policies used by the Naive example.

Pipeline: **rewrite → TF-IDF + BM25 → reciprocal-rank fusion → rerank/filter → compress → answer**.

Everything except the optional final model call uses only the Python standard library.

## 1. Import the reusable implementation

In [ ]:
from pathlib import Path
import json
import os
import sys

ROOT = Path.cwd()
if not (ROOT / "rag_core.py").exists():
    candidates = [ROOT / "advanced-rag", ROOT.parent / "advanced-rag"]
    ROOT = next(path for path in candidates if (path / "rag_core.py").exists())
sys.path.insert(0, str(ROOT))

from rag_core import AdvancedRAG, rewrite_query

print("Project root:", ROOT)

## 2. Initialize the pipeline and inspect the corpus

In [ ]:
rag = AdvancedRAG(ROOT / "documents", top_k=4, candidate_k=10, score_threshold=0.34)
print(f"Documents: {len(rag.documents)}")
print(f"Chunks: {len(rag.chunks)}")
for document in rag.documents:
    print(f"- {document.title}")

## 3. Rewrite an underspecified question

In [ ]:
question = "Can unused leave roll into next year, and is there a deadline?"
rewritten = rewrite_query(question)
print("Original: ", question)
print("Rewritten:", rewritten)

The rewrite is deterministic and visible. In a production system it could be replaced by a model, rules, or domain vocabulary, but it should be evaluated rather than assumed to help.

## 4. Compare the two retrievers

In [ ]:
tfidf_results = rag.tfidf.search(rewritten)[:5]
bm25_results = rag.bm25.search(rewritten)[:5]

print("TF-IDF ranking")
for rank, (chunk, score) in enumerate(tfidf_results, 1):
    print(rank, chunk.id, chunk.title, round(score, 4))

print("\nBM25 ranking")
for rank, (chunk, score) in enumerate(bm25_results, 1):
    print(rank, chunk.id, chunk.title, round(score, 4))

## 5. Fuse, rerank, filter, and compress

In [ ]:
rewritten, reranked, filtered = rag.retrieve(question)

print("Top reranked candidates")
for candidate in reranked[:6]:
    print(candidate.chunk.id, candidate.chunk.title, round(candidate.rerank_score, 4))

print("\nFinal compressed evidence")
for rank, candidate in enumerate(filtered, 1):
    print(f"[{rank}] {candidate.chunk.title} | rerank={candidate.rerank_score:.4f}")
    print(candidate.compressed_text, "\n")

## 6. Generate the answer and inspect the complete trace

In [ ]:
response = rag.ask(question, generator="local")
print("ANSWER")
print(response["answer"])
print("\nREWRITTEN QUERY")
print(response["rewritten_query"])
print("\nTRACE")
print(json.dumps(response["trace"], indent=2))

## 7. Test difficult wording

In [ ]:
questions = [
    "I think an account was breached. How fast must I tell security?",
    "Do I need proof for a R250 taxi on a work trip?",
    "Can I work from another country for a month?",
]

for item in questions:
    result = rag.ask(item, generator="local")
    print("\nQ:", item)
    print("Rewrite:", result["rewritten_query"])
    print("A:", result["answer"])
    print("Top source:", result["sources"][0]["title"])

## 8. Evaluate the complete retrieval pipeline

In [ ]:
evaluation = [
    ("Can unused leave roll over?", "annual-leave-policy"),
    ("Account breach reporting time", "information-security-policy"),
    ("Proof needed for a small travel cost", "travel-expense-policy"),
    ("Paid leave for a secondary parent", "parental-leave-policy"),
    ("Deadline to return a work laptop", "equipment-policy"),
]

correct = 0
for query, expected_document in evaluation:
    _, _, evidence = rag.retrieve(query)
    top = evidence[0].chunk.document_id
    hit = top == expected_document
    correct += int(hit)
    print(f"{'PASS' if hit else 'FAIL'} | {query} → {top}")
print(f"Top-1 accuracy: {correct}/{len(evaluation)} = {correct/len(evaluation):.0%}")

## 9. Optional OpenAI generation

Set `OPENAI_API_KEY` in the environment and pass `generator="openai"`. The local Advanced RAG stages still choose and compress the evidence before the Responses API call.

In [ ]:
if os.environ.get("OPENAI_API_KEY"):
    openai_response = rag.ask(question, generator="openai")
    print(openai_response["answer"])
else:
    print("OPENAI_API_KEY is not set; all retrieval stages and the offline answer ran locally.")

## 10. Deploy the same pipeline

Run `python server.py` for the included browser UI and JSON API, or build the Dockerfile. The deployment exposes the same intermediate trace shown in this notebook.